# Preparacao e limpeza dos dados - OpenFDA Adverse Drug Events

Este notebook executa apenas as etapas de preparacao e limpeza dos dados utilizadas na Entrega 2. O objetivo e transformar os registros brutos em uma base tabular consistente para analise posterior, sem focar na etapa de predicao.


## 1. Carregamento dos arquivos JSON

Os arquivos brutos sao mantidos no diretorio `datasets/`. A leitura percorre todos os arquivos `*.json` disponiveis e concatena os registros da chave `results`.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)

dataset_dir = Path("../datasets")
if not dataset_dir.exists():
    dataset_dir = Path("datasets")

json_paths = sorted(dataset_dir.glob("*.json"))

if not json_paths:
    raise FileNotFoundError(f"Nenhum arquivo .json encontrado em {dataset_dir.resolve()}")

reports_raw = []
for path in json_paths:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    reports_raw.extend(data.get("results", []))

print(f"Arquivos carregados: {len(json_paths)}")
print(f"Registros carregados: {len(reports_raw)}")


Arquivos carregados: 3
Registros carregados: 36000


## 2. Normalizacao e selecao dos atributos principais

A estrutura original do OpenFDA é aninhada. Nesta etapa, os registros são normalizados para tabela e são mantidos os atributos principais do relatorio e do paciente. Os atributos de medicamentos são tratados separadamente na etapa seguinte, pois ficam dentro da lista `patient.drug`.


In [2]:
reports = pd.json_normalize(reports_raw)

selected_columns = [
    "safetyreportid",
    "serious",
    "occurcountry",
    "patient.patientsex",
    "patient.patientonsetage",
    "patient.patientonsetageunit",
]

missing_selected_columns = [c for c in selected_columns if c not in reports.columns]
if missing_selected_columns:
    raise KeyError(f"Colunas esperadas ausentes no dataset: {missing_selected_columns}")

df_raw_selected = reports[selected_columns].copy()
df_raw_selected["safetyreportid"] = df_raw_selected["safetyreportid"].astype("string")

print(f"Dimensao inicial selecionada: {df_raw_selected.shape}")
df_raw_selected.head()


Dimensao inicial selecionada: (36000, 6)


,safetyreportid,serious,occurcountry,patient.patientsex,patient.patientonsetage,patient.patientonsetageunit
0,24916426,1,NZ,2,52,801
1,24917560,1,CA,NaN,NaN,NaN
2,24917803,2,US,1,63,801
3,24917951,2,NaN,NaN,NaN,NaN
4,24920814,1,FR,1,53,801


## 3. Normalizacao dos atributos de medicamentos

Os campos `patient.drug.activesubstance.activesubstancename`, `patient.drug.drugcharacterization` e `patient.drug.medicinalproduct` pertencem a uma lista de medicamentos dentro de cada relatorio. Para manter uma linha por `safetyreportid`, esses valores sao agregados por relatorio.


In [3]:
df_drugs = pd.json_normalize(
    reports_raw,
    record_path=["patient", "drug"],
    meta=["safetyreportid"],
    errors="ignore"
)

drug_columns = [
    "safetyreportid",
    "activesubstance.activesubstancename",
    "drugcharacterization",
    "medicinalproduct",
]

missing_drug_columns = [c for c in drug_columns if c not in df_drugs.columns]
if missing_drug_columns:
    raise KeyError(f"Colunas de medicamento esperadas ausentes no dataset: {missing_drug_columns}")

df_drugs = df_drugs[drug_columns].copy()
df_drugs["safetyreportid"] = df_drugs["safetyreportid"].astype("string")

def normalize_text_value(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if not value:
        return np.nan
    return value.upper()


def join_unique_values(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]
    unique_values = sorted(values.unique())
    if not unique_values:
        return np.nan
    return " | ".join(unique_values)


df_drugs["patient.drug.activesubstance.activesubstancename"] = (
    df_drugs["activesubstance.activesubstancename"].apply(normalize_text_value)
)
df_drugs["patient.drug.medicinalproduct"] = (
    df_drugs["medicinalproduct"].apply(normalize_text_value)
)
df_drugs["patient.drug.drugcharacterization"] = (
    df_drugs["drugcharacterization"].astype("string").str.strip()
)

drug_features = (
    df_drugs
    .groupby("safetyreportid")
    .agg(
        **{
            "patient.drug.activesubstance.activesubstancename": (
                "patient.drug.activesubstance.activesubstancename",
                join_unique_values,
            ),
            "patient.drug.drugcharacterization": (
                "patient.drug.drugcharacterization",
                join_unique_values,
            ),
            "patient.drug.medicinalproduct": (
                "patient.drug.medicinalproduct",
                join_unique_values,
            ),
            "patient.drug.count": ("medicinalproduct", "size"),
        }
    )
    .reset_index()
)

df_raw_selected = df_raw_selected.merge(drug_features, on="safetyreportid", how="left")

drug_text_columns = [
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.drugcharacterization",
    "patient.drug.medicinalproduct",
]

df_raw_selected[drug_text_columns] = df_raw_selected[drug_text_columns].fillna("unknown")
df_raw_selected["patient.drug.count"] = df_raw_selected["patient.drug.count"].fillna(0).astype(int)

print(f"Dimensao apos agregacao dos medicamentos: {df_raw_selected.shape}")
df_raw_selected[["safetyreportid", *drug_text_columns, "patient.drug.count"]].head(30)


Dimensao apos agregacao dos medicamentos: (36000, 10)


,safetyreportid,patient.drug.activesubstance.activesubstancename,patient.drug.drugcharacterization,patient.drug.medicinalproduct,patient.drug.count
0,24916426,ESTRADIOL,1,ESTRADIOL,1
1,24917560,CARBAMAZEPINE | CENOBAMATE | LAMOTRIGINE | OXC...,1 | 2,CARBAMAZEPINE | LAMOTRIGINE | OXCARBAZEPINE | ...,12
2,24917803,DUPILUMAB,1,DUPIXENT,2
3,24917951,TAFAMIDIS,1,VYNDAMAX,1
4,24920814,ACETAMINOPHEN | AMPHOTERICIN B | HEPARIN CALCI...,1 | 2,ACETAMINOPHEN | AMPHOTERICIN B | HEPARIN CALCI...,7
5,24921071,ACETAMINOPHEN | CLONIDINE HYDROCHLORIDE | DULO...,1,ACETAMINOPHEN | CLONIDINE HYDROCHLORIDE | DULO...,8
6,24801764,ACETAMINOPHEN | ATORVASTATIN CALCIUM\EZETIMIBE...,1 | 2,CETUXIMAB | DOLIPRANE | DOXYCYCLINE | HERBALS\...,12
7,24802789,TIRZEPATIDE,1,ZEPBOUND,3
8,24802817,DUPILUMAB,1,DUPIXENT,1
9,24804923,CEFTRIAXONE | METRONIDAZOLE,1 | 2,CEFTRIAXONE | METRONIDAZOLE,2


## 4. Diagnostico inicial dos atributos selecionados

Antes da limpeza, sao calculados os valores ausentes por atributo. Esses indicadores permitem comparar a base antes e depois das transformacoes.


In [4]:
missing_before = (
    df_raw_selected
    .isna()
    .sum()
    .to_frame("missing_count_before")
)
missing_before["missing_percent_before"] = (
    missing_before["missing_count_before"] / len(df_raw_selected) * 100
).round(2)

missing_before


,missing_count_before,missing_percent_before
safetyreportid,0,0.00
serious,0,0.00
occurcountry,3691,10.25
patient.patientsex,6047,16.80
patient.patientonsetage,14080,39.11
patient.patientonsetageunit,14079,39.11
patient.drug.activesubstance.activesubstancename,0,0.00
patient.drug.drugcharacterization,0,0.00
patient.drug.medicinalproduct,0,0.00
patient.drug.count,0,0.00


## 5. Remocao de registros sem pais de ocorrencia

O atributo `occurcountry` identifica o pais em que o evento adverso ocorreu. Como a quantidade de nulos e pequena em relacao ao total da amostra e o campo e relevante para analise, os registros sem pais de ocorrencia sao removidos.


In [5]:
df_clean = df_raw_selected.copy()

rows_before_occurcountry = len(df_clean)
df_clean = df_clean.dropna(subset=["occurcountry"]).copy()
rows_after_occurcountry = len(df_clean)

print(f"Registros antes da remocao: {rows_before_occurcountry}")
print(f"Registros depois da remocao: {rows_after_occurcountry}")
print(f"Registros removidos por occurcountry nulo: {rows_before_occurcountry - rows_after_occurcountry}")


Registros antes da remocao: 36000
Registros depois da remocao: 32309
Registros removidos por occurcountry nulo: 3691


## 6. Conversao e imputacao do sexo do paciente

O campo `patient.patientsex` e codificado pelo OpenFDA. Os codigos sao convertidos para categorias interpretaveis e valores ausentes sao imputados como `unknown`, preservando os registros sem assumir sexo masculino ou feminino.


In [6]:
sex_map = {
    "1": "male",
    "2": "female",
    "0": "unknown",
}

df_clean["patient.patientsex"] = (
    df_clean["patient.patientsex"]
    .astype("string")
    .str.strip()
    .map(sex_map)
    .fillna("unknown")
)

df_clean["patient.patientsex"].value_counts(dropna=False)


patient.patientsex
female     15424
male       11242
unknown     5643
Name: count, dtype: int64

## 7. Conversao da idade para anos

Os atributos `patient.patientonsetage` e `patient.patientonsetageunit` sao usados em conjunto para calcular temporariamente a idade em anos. Esse valor intermediario e usado apenas para criar a faixa etaria calculada.


In [7]:
df_clean["patient.patientonsetage"] = pd.to_numeric(
    df_clean["patient.patientonsetage"],
    errors="coerce"
)


def age_to_years(age, unit):
    if pd.isna(age) or pd.isna(unit):
        return np.nan

    unit = str(unit).strip()

    if unit == "800":      # decade
        return age * 10
    if unit == "801":      # year
        return age
    if unit == "802":      # month
        return age / 12
    if unit == "803":      # week
        return age / 52
    if unit == "804":      # day
        return age / 365
    if unit == "805":      # hour
        return age / (365 * 24)

    return np.nan


age_years_calculated = df_clean.apply(
    lambda row: age_to_years(
        row["patient.patientonsetage"],
        row["patient.patientonsetageunit"]
    ),
    axis=1
)

# Idades fora de uma faixa biologicamente plausivel sao tratadas como ausentes.
age_years_calculated = age_years_calculated.mask(
    (age_years_calculated < 0) | (age_years_calculated > 120)
)

pd.DataFrame({
    "patient.patientonsetage": df_clean["patient.patientonsetage"],
    "patient.patientonsetageunit": df_clean["patient.patientonsetageunit"],
    "age_years_calculated_preview": age_years_calculated,
}).head()


,patient.patientonsetage,patient.patientonsetageunit,age_years_calculated_preview
0,52.0,801,52.0
1,NaN,NaN,NaN
2,63.0,801,63.0
4,53.0,801,53.0
5,52.0,801,52.0


## 8. Criacao da faixa etaria calculada

A coluna `patient.ageGroupCalculated` classifica a idade convertida em anos nas faixas etarias definidas para o projeto. Os nomes foram mantidos em ingles para facilitar o uso posterior como dado categorico.


In [8]:
def calculate_age_group(age_years):
    if pd.isna(age_years):
        return "unknown"
    if 0 <= age_years < 2:
        return "baby_early_childhood"
    if 2 <= age_years < 12:
        return "child"
    if 12 <= age_years < 18:
        return "adolescent"
    if 18 <= age_years < 30:
        return "young_adult"
    if 30 <= age_years < 60:
        return "adult"
    if age_years >= 60:
        return "elderly"
    return "unknown"


df_clean["patient.ageGroupCalculated"] = age_years_calculated.apply(calculate_age_group)

df_clean["patient.ageGroupCalculated"].value_counts(dropna=False)


patient.ageGroupCalculated
unknown                 12726
elderly                  9472
adult                    7271
young_adult              1452
adolescent                617
child                     605
baby_early_childhood      166
Name: count, dtype: int64

## 9. Imputacao da faixa etaria com KNN

Como a idade e um atributo essencial para a analise, os registros sem faixa etaria calculavel sao imputados com KNN. O algoritmo e treinado apenas com registros cuja faixa etaria foi calculada a partir da idade e unidade originais. Em seguida, ele estima a faixa dos registros `unknown` usando pais de ocorrencia, sexo e atributos agregados de medicamento.


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

age_group_before_knn = df_clean["patient.ageGroupCalculated"].copy()
known_age_group_mask = age_group_before_knn != "unknown"
unknown_age_group_mask = age_group_before_knn == "unknown"

df_clean["patient.ageGroupCalculated_was_imputed"] = unknown_age_group_mask.astype(int)

knn_features = [
    "occurcountry",
    "patient.patientsex",
    "patient.drug.drugcharacterization",
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.medicinalproduct",
    "patient.drug.count",
]

categorical_features = [
    "occurcountry",
    "patient.patientsex",
    "patient.drug.drugcharacterization",
]

text_active_substance_feature = "patient.drug.activesubstance.activesubstancename"
text_medicinal_product_feature = "patient.drug.medicinalproduct"
numeric_features = ["patient.drug.count"]

if unknown_age_group_mask.any():
    knn_preprocessor = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
            (
                "active_substance",
                CountVectorizer(max_features=100, token_pattern=r"(?u)\b[\w-]+\b"),
                text_active_substance_feature,
            ),
            (
                "medicinal_product",
                CountVectorizer(max_features=100, token_pattern=r"(?u)\b[\w-]+\b"),
                text_medicinal_product_feature,
            ),
            ("num", StandardScaler(), numeric_features),
        ]
    )

    knn_imputer = Pipeline(
        steps=[
            ("preprocessor", knn_preprocessor),
            (
                "knn",
                KNeighborsClassifier(
                    n_neighbors=5,
                    weights="distance",
                    algorithm="brute",
                    metric="euclidean",
                ),
            ),
        ]
    )

    knn_imputer.fit(
        df_clean.loc[known_age_group_mask, knn_features],
        df_clean.loc[known_age_group_mask, "patient.ageGroupCalculated"],
    )

    set_config(working_memory=32)

    unknown_indices = df_clean.index[unknown_age_group_mask]
    predicted_age_groups = []
    batch_size = 500

    for start in range(0, len(unknown_indices), batch_size):
        batch_indices = unknown_indices[start:start + batch_size]
        batch_predictions = knn_imputer.predict(df_clean.loc[batch_indices, knn_features])
        predicted_age_groups.extend(batch_predictions)

    df_clean.loc[unknown_indices, "patient.ageGroupCalculated"] = predicted_age_groups

print("Distribuicao antes da imputacao KNN:")
print(age_group_before_knn.value_counts(dropna=False))

print("\nDistribuicao depois da imputacao KNN:")
print(df_clean["patient.ageGroupCalculated"].value_counts(dropna=False))

print("\nRegistros com faixa etaria imputada por KNN:")
print(df_clean["patient.ageGroupCalculated_was_imputed"].sum())


Distribuicao antes da imputacao KNN:
patient.ageGroupCalculated
unknown                 12726
elderly                  9472
adult                    7271
young_adult              1452
adolescent                617
child                     605
baby_early_childhood      166
Name: count, dtype: int64

Distribuicao depois da imputacao KNN:
patient.ageGroupCalculated
elderly                 16137
adult                   12583
young_adult              1763
adolescent                801
child                     761
baby_early_childhood      264
Name: count, dtype: int64

Registros com faixa etaria imputada por KNN:
12726


## 10. Base final limpa

A base final preserva os atributos originais relevantes e adiciona as colunas calculadas durante a preparacao. Esta base pode ser usada nas etapas seguintes de transformacao, analise comparativa ou modelagem.


In [13]:
final_columns = [
    "safetyreportid",
    "serious",
    "occurcountry",
    "patient.drug.activesubstance.activesubstancename",
    "patient.drug.drugcharacterization",
    "patient.drug.medicinalproduct",
    "patient.drug.count",
    "patient.patientsex",
    "patient.ageGroupCalculated",
    "patient.ageGroupCalculated_was_imputed",
    "patient.patientonsetage",
    "patient.patientonsetageunit",
]

cleaned_df = df_clean[final_columns].copy()
cleaned_df.to_csv('../outputs/drug_events_cleaned.csv')

print(f"Dimensao final limpa: {cleaned_df.shape}")
cleaned_df.head()


Dimensao final limpa: (32309, 12)


,safetyreportid,serious,occurcountry,patient.drug.activesubstance.activesubstancename,patient.drug.drugcharacterization,patient.drug.medicinalproduct,patient.drug.count,patient.patientsex,patient.ageGroupCalculated,patient.ageGroupCalculated_was_imputed,patient.patientonsetage,patient.patientonsetageunit
0,24916426,1,NZ,ESTRADIOL,1,ESTRADIOL,1,female,adult,0,52.0,801
1,24917560,1,CA,CARBAMAZEPINE | CENOBAMATE | LAMOTRIGINE | OXC...,1 | 2,CARBAMAZEPINE | LAMOTRIGINE | OXCARBAZEPINE | ...,12,unknown,adult,1,NaN,NaN
2,24917803,2,US,DUPILUMAB,1,DUPIXENT,2,male,elderly,0,63.0,801
4,24920814,1,FR,ACETAMINOPHEN | AMPHOTERICIN B | HEPARIN CALCI...,1 | 2,ACETAMINOPHEN | AMPHOTERICIN B | HEPARIN CALCI...,7,male,adult,0,53.0,801
5,24921071,1,AU,ACETAMINOPHEN | CLONIDINE HYDROCHLORIDE | DULO...,1,ACETAMINOPHEN | CLONIDINE HYDROCHLORIDE | DULO...,8,male,adult,0,52.0,801


## 11. Comparacao antes e depois da limpeza

A tabela abaixo resume a quantidade e o percentual de valores ausentes antes e depois das principais transformacoes.


In [11]:
missing_after = (
    cleaned_df
    .isna()
    .sum()
    .to_frame("missing_count_after")
)
missing_after["missing_percent_after"] = (
    missing_after["missing_count_after"] / len(cleaned_df) * 100
).round(2)

comparison = missing_before.join(missing_after, how="outer").fillna(0)
comparison


,missing_count_before,missing_percent_before,missing_count_after,missing_percent_after
occurcountry,3691.0,10.25,0.0,0.0
patient.ageGroupCalculated,0.0,0.00,0.0,0.0
patient.ageGroupCalculated_was_imputed,0.0,0.00,0.0,0.0
patient.drug.activesubstance.activesubstancename,0.0,0.00,0.0,0.0
patient.drug.count,0.0,0.00,0.0,0.0
patient.drug.drugcharacterization,0.0,0.00,0.0,0.0
patient.drug.medicinalproduct,0.0,0.00,0.0,0.0
patient.patientonsetage,14080.0,39.11,0.0,0.0
patient.patientonsetageunit,14079.0,39.11,0.0,0.0
patient.patientsex,6047.0,16.80,0.0,0.0
